# 1. Imports


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn import set_config

import joblib

In [ ]:
set_config(transform_output = "pandas")

# 2. Load Data

In [ ]:
df = pd.read_csv("healthcare-dataset-stroke-data.csv")

### Drop Unnecessary Columns

In [ ]:
df = df.drop(columns = ['id', 'Residence_type'])

# 3. Train Test Split

In [ ]:
X = df.drop('stroke', axis = 1)
y = df['stroke']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

# 4. Feature Engineering Function

In [ ]:
def feature_engineering(X):
  X = X.copy()

  # New feature
  X['is_smoking_unkonwn'] = (X['smoking_status'] == 'Unknown').astype(int)

  # Fix category
  X['work_type'] = X['work_type'].replace('Never_worked', 'children')

  return X

# 5. Feature Groups

In [ ]:
numeric_features = ['age', 'bmi', 'avg_glucose_level']
categorical_features = ['gender', 'hypertension', 'heart_disease', 'ever_married', 'work_type', 'smoking_status']

# 6. Preprocessing

In [ ]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy = 'mean')),
    ('log_transform', FunctionTransformer(np.log1p)),
    ('scaler', StandardScaler())
])

In [ ]:
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy = 'most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown = 'ignore', sparse_output = False))
])

In [ ]:
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

# 7. Final Pipeline

In [ ]:
pipeline = Pipeline([
    ('feature_engineering', FunctionTransformer(feature_engineering, validate = False)),
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(class_weight = 'balanced', max_iter = 1000))
])

# 8. Train

In [ ]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('feature_engineering',
                 FunctionTransformer(func=<function feature_engineering at 0x7fcdc7836fc0>)),
                ('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('log_transform',
                                                                   FunctionTransformer(func=<ufunc 'log1p'>)),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'bmi',
                                                   'avg_glucose_level']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['gender', 'hypertension',
                                                   'heart_disease',
                                                   'ever_married', 'work_type',
                                                   'smoking_status'])])),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

# 9. Evaluation

In [ ]:
y_pred = pipeline.predict(X_test)

In [ ]:
print("=== Classification Report ===")
print(classification_report(y_test, y_pred))

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.99      0.72      0.83       972
           1       0.13      0.84      0.23        50

    accuracy                           0.73      1022
   macro avg       0.56      0.78      0.53      1022
weighted avg       0.95      0.73      0.80      1022



In [ ]:
print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred))

=== Confusion Matrix ===
[[699 273]
 [  8  42]]


# 10. Cross Validation

In [ ]:
cv_scores = cross_val_score(pipeline, X, y, cv = 5, scoring = 'recall')

In [ ]:
print("=== CV Recall ===")
print("Mean Recall: ", cv_scores.mean())

=== CV Recall ===
Mean Recall:  0.839265306122449
